# Music Listening Behavior & Weather Analysis
**DSA 210 - Spring 2026**

Analyzing how weather conditions in Istanbul influence personal Apple Music listening behavior.

## ☁️ Step 0: Mount Google Drive (recommended)
This lets you save results permanently. If you skip this, files are lost when the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change this to wherever you want files saved in your Drive
SAVE_DIR = '/content/drive/MyDrive/DSA210'

import os
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Files will be saved to: {SAVE_DIR}')

## 📂 Step 1: Upload your Apple Music CSV
Run this cell → a file picker will appear → select `Apple_Music_-_Play_History_Daily_Tracks.csv`

In [ ]:
from google.colab import files
uploaded = files.upload()  # opens file picker

# Get the filename that was uploaded
CSV_FILE = list(uploaded.keys())[0]
print(f'Uploaded: {CSV_FILE}')

## 📦 Step 2: Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import os
from scipy.stats import mannwhitneyu, kruskal, spearmanr

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12
})
sns.set_palette('husl')
print('Libraries loaded ✓')

---
## 1. Data Loading & Cleaning

In [ ]:
df_raw = pd.read_csv(CSV_FILE)
print(f'Raw shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
df = df_raw.copy()

# Parse date (YYYYMMDD → datetime)
df['date'] = pd.to_datetime(df['Date Played'].astype(str), format='%Y%m%d')

# Hours column: '16, 17' or '10, 11, 13' → take the first hour
df['hour'] = df['Hours'].astype(str).str.split(',').str[0].str.strip().astype(int)

# Split 'Artist - Track Name'
split = df['Track Description'].str.split(' - ', n=1, expand=True)
df['artist']     = split[0].str.strip()
df['track_name'] = split[1].str.strip()

# Play duration
df['play_duration_sec'] = df['Play Duration Milliseconds'] / 1000
df['play_duration_min'] = df['play_duration_sec'] / 60

# Flags
df['fully_played'] = df['End Reason Type'] == 'NATURAL_END_OF_TRACK'
df['was_skipped']  = df['Skip Count'] > 0

# Rename
df = df.rename(columns={'Play Count':'play_count','Skip Count':'skip_count',
                        'Source Type':'source_type','End Reason Type':'end_reason'})

# Time features
df['year']        = df['date'].dt.year
df['month']       = df['date'].dt.month
df['month_name']  = df['date'].dt.strftime('%b')
df['day_of_week'] = df['date'].dt.dayofweek
df['dow_name']    = df['date'].dt.strftime('%a')

def get_season(m):
    if m in [12,1,2]:   return 'Winter'
    elif m in [3,4,5]:  return 'Spring'
    elif m in [6,7,8]:  return 'Summer'
    else:               return 'Autumn'

df['season'] = df['month'].apply(get_season)

def time_bucket(h):
    if   0 <= h < 6:   return 'Night (0-6)'
    elif 6 <= h < 12:  return 'Morning (6-12)'
    elif 12 <= h < 18: return 'Afternoon (12-18)'
    else:              return 'Evening (18-24)'

df['time_of_day'] = df['hour'].apply(time_bucket)

# Remove rows never actually played
df_played = df[df['play_duration_sec'] > 0].copy()

# Filter to last 2.5 years (October 2023 onwards)
df_played = df_played[df_played['date'] >= '2023-10-01'].copy()

print(f'After removing unplayed rows + date filter: {len(df_played):,} rows')
print('Date range:', df_played['date'].min().date(), '→', df_played['date'].max().date())

---
## 2. Weather Data (Open-Meteo — no API key needed)

In [ ]:
WEATHER_CACHE = os.path.join(SAVE_DIR, 'weather_istanbul.csv')

def fetch_weather(start_date, end_date):
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude': 41.0082, 'longitude': 28.9784,
        'start_date': start_date, 'end_date': end_date,
        'daily': ['temperature_2m_max','temperature_2m_min','temperature_2m_mean',
                  'precipitation_sum','rain_sum','sunshine_duration',
                  'windspeed_10m_max','weathercode'],
        'timezone': 'Europe/Istanbul'
    }
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    return pd.DataFrame(resp.json()['daily'])

if os.path.exists(WEATHER_CACHE):
    print('Loading weather from Drive cache...')
    weather = pd.read_csv(WEATHER_CACHE, parse_dates=['time'])
else:
    print('Fetching from Open-Meteo (~10s)...')
    start = df_played['date'].min().strftime('%Y-%m-%d')
    end   = df_played['date'].max().strftime('%Y-%m-%d')
    weather = fetch_weather(start, end)
    weather['time'] = pd.to_datetime(weather['time'])
    weather.to_csv(WEATHER_CACHE, index=False)
    print('Saved to Drive.')

def wmo_to_condition(code):
    if pd.isna(code):      return 'Unknown'
    code = int(code)
    if code == 0:          return 'Clear'
    elif code in [1,2,3]:  return 'Partly Cloudy'
    elif code in [45,48]:  return 'Foggy'
    elif 51 <= code <= 67: return 'Rainy'
    elif 71 <= code <= 77: return 'Snowy'
    elif 80 <= code <= 82: return 'Showers'
    elif 95 <= code <= 99: return 'Thunderstorm'
    else:                  return 'Other'

weather['condition']      = weather['weathercode'].apply(wmo_to_condition)
weather['sunshine_hours'] = weather['sunshine_duration'] / 3600
weather = weather.rename(columns={'time': 'date'})

print(f'Weather rows: {len(weather):,}')
weather.head(3)

In [ ]:
# Merge music + weather
df_merged = df_played.merge(
    weather[['date','temperature_2m_mean','temperature_2m_max','temperature_2m_min',
             'precipitation_sum','sunshine_hours','condition']],
    on='date', how='left'
)
print(f'Merged shape: {df_merged.shape}')

---
## 3. Daily Aggregation

In [ ]:
daily = df_merged.groupby('date').agg(
    n_plays            = ('track_name',          'count'),
    total_listen_min   = ('play_duration_min',    'sum'),
    total_skips        = ('skip_count',           'sum'),
    fraction_complete  = ('fully_played',         'mean'),
    temp_mean          = ('temperature_2m_mean',  'first'),
    temp_max           = ('temperature_2m_max',   'first'),
    precipitation      = ('precipitation_sum',    'first'),
    sunshine_hours     = ('sunshine_hours',       'first'),
    condition          = ('condition',            'first'),
    season             = ('season',               'first'),
    day_of_week        = ('day_of_week',          'first'),
    dow_name           = ('dow_name',             'first'),
    month              = ('month',                'first'),
    month_name         = ('month_name',           'first'),
    year               = ('year',                 'first'),
).reset_index()

daily['skip_rate'] = daily['total_skips'] / daily['n_plays']
daily['is_rainy']  = daily['precipitation'] > 1.0
daily['is_sunny']  = daily['sunshine_hours'] > 6.0

print(f'Daily summary: {len(daily):,} days')
daily.head(3)

---
## 4. EDA

In [ ]:
# Yearly plays
yearly = daily.groupby('year')['n_plays'].sum()
fig, ax = plt.subplots()
bars = ax.bar(yearly.index, yearly.values, color=sns.color_palette('husl', len(yearly)))
ax.bar_label(bars, fmt='%d')
ax.set(title='Total Plays Per Year', xlabel='Year', ylabel='Number of Plays')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig_yearly_plays.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Monthly average plays
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly_avg = daily.groupby('month_name')['n_plays'].mean().reindex(month_order)

fig, ax = plt.subplots()
ax.plot(month_order, monthly_avg.values, marker='o', linewidth=2, color='steelblue')
ax.fill_between(range(12), monthly_avg.values, alpha=0.15, color='steelblue')
ax.set_xticks(range(12)); ax.set_xticklabels(month_order)
ax.set(title='Average Daily Plays by Month', xlabel='Month', ylabel='Avg Plays per Day')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig_monthly_plays.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plays by hour of day
hourly = df_played.groupby('hour').size()
fig, ax = plt.subplots()
ax.bar(hourly.index, hourly.values, color='coral')
ax.set(title='Listening Activity by Hour of Day', xlabel='Hour', ylabel='Number of Plays')
ax.set_xticks(range(0,24))
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig_hourly_plays.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plays by day of week
dow_order = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow_avg = daily.groupby('dow_name')['n_plays'].mean().reindex(dow_order)

fig, ax = plt.subplots(figsize=(8,4))
ax.bar(dow_order, dow_avg.values,
       color=['#4C72B0' if d not in ['Sat','Sun'] else '#DD8452' for d in dow_order])
ax.set(title='Average Plays by Day of Week', xlabel='Day', ylabel='Avg Plays per Day')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig_dow_plays.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top 15 artists
top_artists = df_played['artist'].value_counts().head(15)
fig, ax = plt.subplots(figsize=(10,5))
top_artists.sort_values().plot.barh(ax=ax, color='mediumpurple')
ax.set(title='Top 15 Most Played Artists', xlabel='Total Plays')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig_top_artists.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Listening time vs temperature scatter
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(daily['temp_mean'], daily['total_listen_min'], alpha=0.2, s=10, color='steelblue')
valid = daily[['temp_mean','total_listen_min']].dropna()
m, b = np.polyfit(valid['temp_mean'], valid['total_listen_min'], 1)
x_line = np.linspace(valid['temp_mean'].min(), valid['temp_mean'].max(), 100)
axes[0].plot(x_line, m*x_line+b, 'r--', linewidth=2)
axes[0].set(title='Listen Time vs Temperature', xlabel='Mean Temp (°C)', ylabel='Minutes/Day')

axes[1].scatter(daily['precipitation'], daily['total_listen_min'], alpha=0.2, s=10, color='coral')
axes[1].set(title='Listen Time vs Precipitation', xlabel='Precipitation (mm)', ylabel='Minutes/Day')
axes[1].set_xlim(-1, 50)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig_weather_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Average plays by weather condition
cond_plays = daily.groupby('condition')['n_plays'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9, 4))
cond_plays.plot.bar(ax=ax, color=sns.color_palette('Set2', len(cond_plays)))
ax.set(title='Average Daily Plays by Weather Condition', xlabel='', ylabel='Avg Plays per Day')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig_plays_by_condition.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Hypothesis Tests

In [ ]:
def report(hypothesis, test_name, stat, p_value, alpha=0.05, extra=''):
    sig = 'SIGNIFICANT ✓' if p_value < alpha else 'not significant'
    print(f'\n{"="*60}')
    print(f'{hypothesis}')
    print(f'Test: {test_name}')
    print(f'Statistic = {stat:.4f} | p-value = {p_value:.4f}')
    print(f'Result (α=0.05): {sig}')
    if extra: print(extra)

In [ ]:
# H1: Rainy vs non-rainy → listening duration
rainy  = daily[daily['is_rainy']]['total_listen_min'].dropna()
other  = daily[~daily['is_rainy']]['total_listen_min'].dropna()
stat, p = mannwhitneyu(rainy, other, alternative='two-sided')
report('H1: Do rainy days lead to more listening time?', 'Mann-Whitney U',
       stat, p, extra=f'  Rainy median: {rainy.median():.1f} min | Non-rainy median: {other.median():.1f} min')

In [ ]:
# H2: Temperature ↔ listening duration
valid = daily[['temp_mean','total_listen_min']].dropna()
corr, p = spearmanr(valid['temp_mean'], valid['total_listen_min'])
report('H2: Does temperature correlate with daily listening duration?',
       'Spearman Correlation', corr, p, extra=f'  ρ = {corr:.3f}')

In [ ]:
# H3: Listening activity across seasons
season_order = ['Spring','Summer','Autumn','Winter']
groups = [daily[daily['season']==s]['n_plays'].dropna() for s in season_order]
stat, p = kruskal(*groups)
medians = {s: daily[daily['season']==s]['n_plays'].median() for s in season_order}
report('H3: Does listening activity differ across seasons?', 'Kruskal-Wallis',
       stat, p, extra='  Medians: ' + ', '.join(f'{s}={v:.1f}' for s,v in medians.items()))

In [ ]:
# H4: Weekend vs weekday skip rate
weekend = daily[daily['day_of_week'].isin([5,6])]['skip_rate'].dropna()
weekday = daily[~daily['day_of_week'].isin([5,6])]['skip_rate'].dropna()
stat, p = mannwhitneyu(weekend, weekday, alternative='two-sided')
report('H4: Do weekends have a different skip rate than weekdays?', 'Mann-Whitney U',
       stat, p, extra=f'  Weekend median: {weekend.median():.3f} | Weekday median: {weekday.median():.3f}')

In [ ]:
# H5: Precipitation ↔ number of plays
valid2 = daily[['precipitation','n_plays']].dropna()
corr, p = spearmanr(valid2['precipitation'], valid2['n_plays'])
report('H5: Does precipitation correlate with number of plays per day?',
       'Spearman Correlation', corr, p, extra=f'  ρ = {corr:.3f}')

In [ ]:
# Save cleaned datasets to Drive for later use
df_merged.to_csv(os.path.join(SAVE_DIR, 'music_weather_merged.csv'), index=False)
daily.to_csv(os.path.join(SAVE_DIR, 'music_weather_daily.csv'), index=False)
print('Saved to Google Drive ✓')
print('  → music_weather_merged.csv  (track-level)')
print('  → music_weather_daily.csv   (daily aggregates)')